In [2]:
import pandas as pd
import sqlite3
import os
import sys
import platform
from rich import print

In [1]:
#To Do - Recognise and Tag Named Entities in the text of Dundee and Natural Stories



In [3]:
#For each example in the blimp dataset, find the length and save 
#For examples with the two sentences having different lengths, decide what to do (easier is first to just save both the lengths separately, can always be combined later)

if "pop-os" in platform.node():
    ROOT = r"/home/abishekthamma/PycharmProjects/masters_thesis/ss-llm/nanoGPT/"
else:
    ROOT = r'/gpfs/home4/athamma/repo/ss-llm/nanoGPT/'#Given a set of details, return a dataframe compiled with the details

RESULTS_ROOT = os.path.join(ROOT, "results")
SQL_DB = os.path.join(RESULTS_ROOT, "results.db")


def create_connection_cursor(db_file):
    """
    Create a database connection to the SQLite database specified by the db_file

    Args:
        db_file (str): database file

    Returns:
        Connection object or None
    """
    conn = sqlite3.connect(db_file)
    c = conn.cursor()
    return conn, c

conn, c = create_connection_cursor(SQL_DB)


#Other Utils
import io
import sys
import contextlib

@contextlib.contextmanager
def silence_prints():
    sys.stdout, old = io.StringIO(), sys.stdout
    try:
        yield
    finally:
        sys.stdout = old


In [4]:
story_df = pd.read_sql_query("SELECT StoryWordID, CorpusID, StoryID, WordID, Word from Story", conn)

story_df

,StoryWordID,CorpusID,StoryID,WordID,Word
0,1,1,1,1,If
1,2,1,1,2,you
2,3,1,1,3,were
3,4,1,1,4,to
4,5,1,1,5,journey
...,...,...,...,...,...
61752,61753,2,20,2449,countries
61753,61754,2,20,2450,are
61754,61755,2,20,2451,pushing
61755,61756,2,20,2452,ahead


In [5]:
story_df["end_of_sentence"] = story_df["Word"].apply(lambda x: True if any([x.endswith(y) for y in [".", "!", "?"]]) else False)

#Adding sentence ID for each story in each corpus

story_df["sentence_id"] = story_df.groupby(["CorpusID", "StoryID"])["end_of_sentence"].cumsum()

#Updating the sentence ID for the last word in each story to be the same as its previous word

story_df["sentence_id"] = story_df["sentence_id"].where(~story_df["end_of_sentence"], story_df["sentence_id"] - 1)


story_df

,StoryWordID,CorpusID,StoryID,WordID,Word,end_of_sentence,sentence_id
0,1,1,1,1,If,False,0
1,2,1,1,2,you,False,0
2,3,1,1,3,were,False,0
3,4,1,1,4,to,False,0
4,5,1,1,5,journey,False,0
...,...,...,...,...,...,...,...
61752,61753,2,20,2449,countries,False,110
61753,61754,2,20,2450,are,False,110
61754,61755,2,20,2451,pushing,False,110
61755,61756,2,20,2452,ahead,False,110


In [6]:
import spacy
from spacy import displacy
from collections import Counter

nlp = spacy.load("en_core_web_sm")


In [ ]:
from spacy.tokens import Doc
from tqdm import tqdm
tqdm.pandas()

def get_named_entities(sentence_subset_df):

    sentence_custom_tokenized = sentence_subset_df["Word"].tolist()
    if len(sentence_custom_tokenized) == 1:
        if sentence_custom_tokenized[0] == ".":
            sentence_custom_tokenized = sentence_custom_tokenized
    else:
        sentence_custom_tokenized[-1] = sentence_custom_tokenized[-1][:-1]

    spaces = [True] * (len(sentence_custom_tokenized)-1) + [False]
    try:
            
        doc = Doc(nlp.vocab, words=sentence_custom_tokenized, spaces=spaces)

        for name, proc in nlp.pipeline:
            doc = proc(doc)
    except Exception as e:
        print(f"Error in processing {sentence_subset_df}")
        print(e)
        raise e
    return [word.ent_type_ for word in doc]
    # NER_flag = []
    
    # #For each word in the sentence, check if it is a named entity
    # for word in doc:
    #     if word.ent_type_:
    #         NER_flag.append(True)
    #     else:
    #         NER_flag.append(False)
    
    # return NER_flag


story_df["NER_flag"] = story_df.groupby(["CorpusID", "StoryID", "sentence_id"]).progress_apply(get_named_entities).explode().tolist()
story_df["is_NER"] = story_df["NER_flag"].apply(lambda x: True if x else False)

story_df

100%|██████████| 2734/2734 [00:17<00:00, 158.85it/s]


,StoryWordID,CorpusID,StoryID,WordID,Word,end_of_sentence,sentence_id,NER_flag
0,1,1,1,1,If,False,0,
1,2,1,1,2,you,False,0,
2,3,1,1,3,were,False,0,
3,4,1,1,4,to,False,0,
4,5,1,1,5,journey,False,0,
...,...,...,...,...,...,...,...,...
61752,61753,2,20,2449,countries,False,110,
61753,61754,2,20,2450,are,False,110,
61754,61755,2,20,2451,pushing,False,110,
61755,61756,2,20,2452,ahead,False,110,


In [ ]:
# for row in tqdm(story_df[["StoryWordID", "NER_flag"]].to_dict(orient="records")):
#     c.execute("UPDATE Story SET NERTag = ? WHERE StoryWordID = ?", (row["NER_flag"], row["StoryWordID"]))

# conn.commit()

In [27]:
# #For first 4 sentences in the first story of the first corpus get the named entities and print them

# for sentence_df in story_df[(story_df["CorpusID"] == 1) & (story_df["StoryID"] == 1) & (story_df["sentence_id"] < 5)].groupby(["CorpusID", "StoryID", "sentence_id"]):
#     ner_sentence = get_named_entities(sentence_df[1])

#     print([(word, ner) for word, ner in zip(sentence_df[1]["Word"].tolist(), ner_sentence)])
